# 서울 교통카드 Trip 레코드 (노인) → DuckDB 적재

공공데이터포털 `RegionalTransportationCardUsageSyntheticData/getSeoulTransportationCardUsageSyntheticData`.

**Trip-level 데이터**: 개별 탑승 이벤트 (가상 카드번호 포함).

- `opr_ymd` = **20260222** (일요일, 주말 패턴 파악)
- `users_type_cd` = **04 (경로/노인)** — API가 필수 필터
- `rte_id` 필수 → **서울 1,157 노선 순회**
- 응답 속도 ~0.4초/건 → **예상 10분**
- 테이블: `elderly_card_trips`
- PK: (vr_card_no, ride_dt, rte_id) — 한 카드의 한 승차시각은 유일

### 분석 활용
- **OD 매트릭스**: ride_sttn_id → goff_sttn_id
- **노인 장거리 이동**: utztn_dstnc
- **환승 빈도**: trnf_cnt
- **주말 이동 패턴**: 쇼핑/병원/경로당 방문 등

In [1]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 64)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [2]:
# ============================================================
# 셀 2 - Pydantic 모델
# ============================================================
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class CardTripItem(BaseModel):
    """개별 카드 탑승 trip 한 건.

    PK: (vr_card_no, ride_dt, rte_id)
    ⚠️ 공공데이터 API 특성상 PK 필드도 null 올 수 있음 → 전부 Optional로 방어.
       적재 단계(셀 5)에서 PK null 행은 제외 (DuckDB PK는 NULL 불가).

    필드 의미:
      - vr_card_no: 가상 카드번호 (개별 승객 추적용, 원본 카드번호 아님)
      - ride_dt/goff_dt: yyyyMMddHHmmss 14자리 (초 단위)
      - ride/goff_sttn_id: 승차/하차 정류소 ID
      - utztn_dstnc: 이용 거리(m)
      - brdg_hr: 승차 시간(초)
      - trnf_cnt: 환승 횟수
      - msv_intrpl_yn: 누락 대체 여부
    """
    # PK 후보 (Optional로 방어, 적재 시 null 행 제외)
    vr_card_no: Optional[str] = None
    ride_dt: Optional[str] = None
    rte_id: Optional[str] = None

    # 메타
    opr_ymd: Optional[str] = None
    users_type_cd: Optional[str] = None

    # OD
    ride_ctpv_cd: Optional[str] = None
    goff_ctpv_cd: Optional[str] = None
    ride_sttn_id: Optional[str] = None
    goff_sttn_id: Optional[str] = None
    goff_dt: Optional[str] = None

    # 사업자/카드/교통수단
    clcln_bzmn_id: Optional[str] = None
    card_se_cd: Optional[str] = None
    clcln_bzmn_trfc_mns_cd: Optional[str] = None

    # 지표
    trnf_cnt: int = 0
    utztn_nope: int = 0
    utztn_dstnc: int = 0
    brdg_hr: int = 0
    msv_intrpl_yn: Optional[str] = None


print("모델 로드 완료")

모델 로드 완료


In [4]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
CARD_TRIPS_URL = "https://apis.data.go.kr/1613000/RegionalTransportationCardUsageSyntheticData/getSeoulTransportationCardUsageSyntheticData"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집."""
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=30)
        res.raise_for_status()
        payload = res.json()

        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.05)

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [7]:
# ============================================================
# 셀 4 - 파라미터 준비: 서울 1,157 노선 + 테이블 + 재개
# ============================================================
# API 파라미터: opr_ymd + rte_id + users_type_cd 모두 필수
# → route_station에서 서울 고유 rte_id 추출해서 루프

OPR_YMD = "20260222"   # 일요일 (주말 패턴)
USERS_TYPE = "04"      # 경로/노인

print(f"대상 날짜: {OPR_YMD} (일요일)")
print(f"이용자유형: {USERS_TYPE} (경로)")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS elderly_card_trips (
        vr_card_no               TEXT,   -- 가상 카드번호
        ride_dt                  TEXT,   -- 승차 시각 (yyyyMMddHHmmss)
        rte_id                   TEXT,   -- 노선 ID
        opr_ymd                  TEXT,
        users_type_cd            TEXT,
        ride_ctpv_cd             TEXT,
        goff_ctpv_cd             TEXT,
        ride_sttn_id             TEXT,
        goff_sttn_id             TEXT,
        goff_dt                  TEXT,
        clcln_bzmn_id            TEXT,
        card_se_cd               TEXT,
        clcln_bzmn_trfc_mns_cd   TEXT,
        trnf_cnt                 INTEGER,
        utztn_nope               INTEGER,
        utztn_dstnc              INTEGER,  -- 이용거리(m)
        brdg_hr                  INTEGER,  -- 승차시간(초)
        msv_intrpl_yn            TEXT,
        PRIMARY KEY (vr_card_no, ride_dt, rte_id)
    )
    """)

with db_open(read_only=True) as con:
    route_df = con.execute("""
        SELECT DISTINCT rte_id,
               ANY_VALUE(rte_no) AS rte_no,
               ANY_VALUE(rte_nm) AS rte_nm
        FROM route_station
        WHERE ctpv_cd = '11'
        GROUP BY rte_id
        ORDER BY rte_id
    """).df()

    done_df = con.execute("""
        SELECT DISTINCT rte_id
        FROM elderly_card_trips
        WHERE opr_ymd = ? AND users_type_cd = ?
    """, [OPR_YMD, USERS_TYPE]).df()

done_set = set(done_df["rte_id"]) if len(done_df) > 0 else set()
remaining = route_df[~route_df["rte_id"].isin(done_set)].reset_index(drop=True)

print(f"\n서울 노선: {len(route_df)}개")
print(f"이미 완료: {len(route_df) - len(remaining)}개")
print(f"남은 작업: {len(remaining)}개")

대상 날짜: 20260222 (일요일)
이용자유형: 04 (경로)

서울 노선: 1157개
이미 완료: 947개
남은 작업: 210개


In [5]:
# ============================================================
# 셀 5 - 서울 노선 순회, 노선 단위 즉시 DB 저장
# ============================================================
# ⚠️ PK 필드(vr_card_no, ride_dt, rte_id)가 null인 행은 제외 (DuckDB PK 제약)
# 누적 null 건수는 cumul_null_dropped로 추적

failed: list[tuple[str, str, str]] = []    # (rte_id, rte_nm, error)
started_at = datetime.now()
processed_total = 0
cumul_null_dropped = 0

for i, row in enumerate(remaining.itertuples(index=False), start=1):
    rte_id = row.rte_id
    rte_nm = row.rte_nm or "(이름없음)"

    try:
        items = fetch_all_pages(
            CARD_TRIPS_URL,
            CardTripItem,
            extra_params={
                "opr_ymd":       OPR_YMD,
                "rte_id":        rte_id,
                "users_type_cd": USERS_TYPE,
            },
        )

        if items:
            df_rows = pd.DataFrame([it.model_dump() for it in items])

            # PK null 행 제외 (DuckDB PK는 NULL 허용 안 함)
            before_n = len(df_rows)
            df_rows = df_rows.dropna(subset=["vr_card_no", "ride_dt", "rte_id"])
            null_dropped = before_n - len(df_rows)
            cumul_null_dropped += null_dropped

            df_rows = df_rows.drop_duplicates(
                subset=["vr_card_no", "ride_dt", "rte_id"]
            )
        else:
            df_rows = pd.DataFrame()

        if len(df_rows) > 0:
            with db_open() as con:
                con.register("df_view", df_rows)
                con.execute("""
                    INSERT INTO elderly_card_trips
                    SELECT vr_card_no, ride_dt, rte_id, opr_ymd, users_type_cd,
                           ride_ctpv_cd, goff_ctpv_cd, ride_sttn_id, goff_sttn_id, goff_dt,
                           clcln_bzmn_id, card_se_cd, clcln_bzmn_trfc_mns_cd,
                           trnf_cnt, utztn_nope, utztn_dstnc, brdg_hr, msv_intrpl_yn
                    FROM df_view
                    ON CONFLICT (vr_card_no, ride_dt, rte_id) DO NOTHING
                """)
                con.unregister("df_view")

        processed_total += len(df_rows)
        status = f"+{len(df_rows):>4,}건"
    except Exception as e:
        failed.append((rte_id, rte_nm, str(e)[:100]))
        status = f"FAILED ({type(e).__name__})"

    if i % 10 == 0 or i == len(remaining):
        elapsed = datetime.now() - started_at
        pct = i / len(remaining) * 100
        eta_sec = elapsed.total_seconds() / i * (len(remaining) - i)
        print(f"[{i:4d}/{len(remaining)}] {pct:5.1f}% | {rte_id} {rte_nm[:18]:18s} {status:14s} "
              f"| 누적 {processed_total:>7,} | 경과 {str(elapsed).split('.')[0]} | 잔여 ~{int(eta_sec//60)}분")

    time.sleep(0.05)

print(f"\n✅ 수집 완료")
print(f"   이번 실행 적재: {processed_total:,}건")
if cumul_null_dropped > 0:
    print(f"   PK null 제외: {cumul_null_dropped:,}건")
print(f"   실패 노선: {len(failed)}개")
print(f"   총 소요시간: {datetime.now() - started_at}")

with db_open(read_only=True) as con:
    grand_total = con.execute("SELECT COUNT(*) FROM elderly_card_trips").fetchone()[0]
print(f"\n📊 elderly_card_trips 총 누적: {grand_total:,}건")

[  10/337]   3.0% | 11111171 성동13(옥수역.한강버스선착장성수 +   0건         | 누적       0 | 경과 0:00:16 | 잔여 ~8분
[  20/337]   5.9% | 12120096 6100(수락)           +   0건         | 누적       0 | 경과 0:00:33 | 잔여 ~8분
[  30/337]   8.9% | 12126003 6003번(대림역인천공항)     +   0건         | 누적       0 | 경과 0:00:44 | 잔여 ~7분
[  40/337]  11.9% | 12126015 6015번(명동)          +   0건         | 누적       0 | 경과 0:00:59 | 잔여 ~7분
[  50/337]  14.8% | 28001001 9100               +   0건         | 누적       0 | 경과 0:01:11 | 잔여 ~6분
[  60/337]  17.8% | 28314001 1601               +   0건         | 누적       0 | 경과 0:01:26 | 잔여 ~6분
[  70/337]  20.8% | 28357900 M6450              +   0건         | 누적       0 | 경과 0:01:41 | 잔여 ~6분
[  80/337]  23.7% | 41002089 1005               +   0건         | 누적       0 | 경과 0:01:59 | 잔여 ~6분
[  90/337]  26.7% | 41003717 9600               +   0건         | 누적       0 | 경과 0:02:11 | 잔여 ~5분
[ 100/337]  29.7% | 41005014 93                 +   0건         | 누적       0 | 경과 0:02:19 | 잔여 ~5분
[ 110/337]  32.6% | 

In [6]:
# ============================================================
# 셀 6 - (선택) 테이블 초기화
# ============================================================
RESET = False

if RESET:
    with db_open() as con:
        con.execute("DELETE FROM elderly_card_trips")
        cnt = con.execute("SELECT COUNT(*) FROM elderly_card_trips").fetchone()[0]
    print(f"⚠️  초기화 완료: {cnt}건")
else:
    print("RESET=False → 아무 작업 안 함")

RESET=False → 아무 작업 안 함


In [7]:
# ============================================================
# 셀 7 - 실패 노선 재시도
# ============================================================
# 셀 5와 동일하게 PK null 행 제외 (DuckDB PK 제약)
if not failed:
    print("실패 노선 없음")
else:
    print(f"재시도할 실패 노선: {len(failed)}개")
    retry_failed = []
    retry_null_dropped = 0

    for rte_id, rte_nm, prev_err in failed:
        try:
            items = fetch_all_pages(
                CARD_TRIPS_URL, CardTripItem,
                extra_params={"opr_ymd": OPR_YMD, "rte_id": rte_id, "users_type_cd": USERS_TYPE},
            )
            if items:
                df_rows = pd.DataFrame([it.model_dump() for it in items])

                # PK null 행 제외
                before_n = len(df_rows)
                df_rows = df_rows.dropna(subset=["vr_card_no", "ride_dt", "rte_id"])
                retry_null_dropped += before_n - len(df_rows)

                df_rows = df_rows.drop_duplicates(
                    subset=["vr_card_no", "ride_dt", "rte_id"]
                )

                if len(df_rows) > 0:
                    with db_open() as con:
                        con.register("df_view", df_rows)
                        con.execute("""
                            INSERT INTO elderly_card_trips
                            SELECT vr_card_no, ride_dt, rte_id, opr_ymd, users_type_cd,
                                   ride_ctpv_cd, goff_ctpv_cd, ride_sttn_id, goff_sttn_id, goff_dt,
                                   clcln_bzmn_id, card_se_cd, clcln_bzmn_trfc_mns_cd,
                                   trnf_cnt, utztn_nope, utztn_dstnc, brdg_hr, msv_intrpl_yn
                            FROM df_view
                            ON CONFLICT (vr_card_no, ride_dt, rte_id) DO NOTHING
                        """)
                        con.unregister("df_view")
                print(f"  ✅ {rte_id} {rte_nm[:15]}: +{len(df_rows)}건")
        except Exception as e:
            retry_failed.append((rte_id, rte_nm, str(e)[:100]))
            print(f"  ❌ {rte_id}: {e}")
        time.sleep(0.2)

    failed = retry_failed
    print(f"\n재시도 후 여전히 실패: {len(failed)}개")
    if retry_null_dropped > 0:
        print(f"재시도 중 PK null 제외: {retry_null_dropped:,}건")


실패 노선 없음


In [8]:
# ============================================================
# 셀 8 - 검증 + 핵심 분석 쿼리
# ============================================================
with db_open(read_only=True) as con:
    print("=== 기본 통계 ===")
    print(con.execute("""
        SELECT opr_ymd,
               COUNT(*)                  AS 총_trip,
               COUNT(DISTINCT vr_card_no) AS 고유_노인수,
               COUNT(DISTINCT rte_id)    AS 노선수,
               COUNT(DISTINCT ride_sttn_id) AS 승차정류소수,
               ROUND(AVG(utztn_dstnc), 0) AS 평균이동거리_m,
               ROUND(AVG(brdg_hr)/60.0, 1) AS 평균승차분
        FROM elderly_card_trips
        GROUP BY opr_ymd
    """).df())

    # 시간대별 승차 패턴
    print("\n=== 노인 시간대별 승차 (주말) ===")
    print(con.execute("""
        SELECT substr(ride_dt, 9, 2) AS 시간대,
               COUNT(*) AS 승차수
        FROM elderly_card_trips
        GROUP BY 1 ORDER BY 1
    """).df())

    # 노인 승차 TOP 20 정류소
    print("\n=== 노인 승차 TOP 20 정류소 ===")
    print(con.execute("""
        SELECT ride_sttn_id,
               ANY_VALUE(rs.sttn_nm)  AS 정류소명,
               ANY_VALUE(r.locatadd_nm) AS emd_nm,
               COUNT(*) AS 승차수
        FROM elderly_card_trips ect
        LEFT JOIN route_station rs ON rs.sttn_id = ect.ride_sttn_id
        LEFT JOIN region r         ON rs.emd_cd = r.region_cd
        GROUP BY ride_sttn_id
        ORDER BY 승차수 DESC
        LIMIT 20
    """).df())

    # 환승 빈도
    print("\n=== 노인 환승 횟수 분포 ===")
    print(con.execute("""
        SELECT trnf_cnt AS 환승횟수, COUNT(*) AS trip수,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1) AS pct
        FROM elderly_card_trips
        GROUP BY trnf_cnt
        ORDER BY trnf_cnt
    """).df())

=== 기본 통계 ===
    opr_ymd  총_trip  고유_노인수  노선수  승차정류소수  평균이동거리_m  평균승차분
0  20260222   57038   42761  947    7575    3214.0   11.4

=== 노인 시간대별 승차 (주말) ===
   시간대   승차수
0   00   216
1   01    38
2   02    32
3   03    77
4   04   322
5   05  1020
6   06  1327
7   07  1392
8   08  2412
9   09  3286
10  10  4026
11  11  3341
12  12  3934
13  13  4258
14  14  4207
15  15  4253
16  16  4204
17  17  4210
18  18  3697
19  19  3173
20  20  2740
21  21  2301
22  22  1676
23  23   896

=== 노인 승차 TOP 20 정류소 ===
   ride_sttn_id             정류소명             emd_nm    승차수
0       4117121         잠실광역환승센터      서울특별시 송파구 잠실동  12660
1       0070069      청량리.청과물도매시장     서울특별시 동대문구 제기동   3768
2       8001829         청량리역환승센터     서울특별시 동대문구 전농동   3696
3       8000547           미아사거리역      서울특별시 강북구 미아동   3151
4       8001908            고속터미널      서울특별시 서초구 잠원동   2760
5       8000781         청량리역환승센터     서울특별시 동대문구 전농동   2691
6       8501293          청량리수산시장     서울특별시 동대문구 용두동   2604
7       0070707       

---

## 지하철 노선 추가 수집

같은 API가 **지하철 rte_id도 지원**한다는 사실 확인 (실제 호출 테스트 완료).

- `railway_line` 51개 중 **데이터 있는 노선 25개** (나머지 26개는 부산/대구/인천/지방경전철 → NO_DATA — 본 API는 서울권 합성데이터만 제공)
- 노인 trip 합계 **약 45만건** (버스 5만건의 9배 — 노인 장거리 이동의 핵심)
- 기존 `elderly_card_trips` 테이블에 그대로 INSERT (PK 충돌 없음: 버스 `11110001`, 지하철 `002` 형식 다름)
- `clcln_bzmn_trfc_mns_cd`로 운영주체 식별 가능 (200번대=도시철도, `trfc_mns` 테이블 참조)

### 같은 노선이 여러 rte_id로 분리된 사례 (실측 확인)
| rte_id | 노선명 | 실측 `trfc_mns_cd` | 운영주체 |
|---|---|---|---|
| 001 | 1호선 | 201 | 서울메트로 (=서울교통공사) |
| 101, 102, 103 | 1호선 | 202 | 한국철도공사 (코레일) |
| 401 | 9호선 | 205 | 9호선 운영주체 |
| 404 | 9호선 | 201 | 서울메트로 코드 (의미 미확인 — TODO 검증) |
| 108, 110 | 경의중앙선 | 202 | 둘 다 코레일 (분리 사유 미확인) |

→ 분석 시 rte_id가 아니라 `clcln_bzmn_trfc_mns_cd` 단위로 운영주체 집계하는 게 안전.

예상 수집 시간: **~15분** (25개 노선, 평균 ~18,000건/노선)

In [8]:
# ============================================================
# 셀 10 - 지하철: 파라미터 + 재개 목록
# ============================================================
# railway_line 51개 중 수도권만 추리고, 이미 적재된 rte_id는 제외
# (NO_DATA 노선은 fetch_all_pages가 빈 리스트 반환 → 자동 skip)

print(f"대상 날짜: {OPR_YMD} (일요일)")
print(f"이용자유형: {USERS_TYPE} (경로)")

with db_open(read_only=True) as con:
    rail_df = con.execute("""
        SELECT DISTINCT rte_id,
               ANY_VALUE(rte_nm) AS rte_nm
        FROM railway_line
        WHERE sarea_nm = '수도권'
        GROUP BY rte_id
        ORDER BY rte_id
    """).df()

    done_df = con.execute("""
        SELECT DISTINCT rte_id
        FROM elderly_card_trips
        WHERE opr_ymd = ? AND users_type_cd = ?
    """, [OPR_YMD, USERS_TYPE]).df()

done_set = set(done_df["rte_id"]) if len(done_df) > 0 else set()
rail_remaining = rail_df[~rail_df["rte_id"].isin(done_set)].reset_index(drop=True)

print(f"\n수도권 지하철 노선: {len(rail_df)}개")
print(f"이미 완료: {len(rail_df) - len(rail_remaining)}개")
print(f"남은 작업: {len(rail_remaining)}개")
print(f"  ※ 일부 노선은 API에서 NO_DATA (지방 운영) → 빈 결과로 skip 됨")

대상 날짜: 20260222 (일요일)
이용자유형: 04 (경로)

수도권 지하철 노선: 40개
이미 완료: 19개
남은 작업: 21개
  ※ 일부 노선은 API에서 NO_DATA (지방 운영) → 빈 결과로 skip 됨


In [9]:
# ============================================================
# 셀 11 - 지하철 노선 순회, 노선 단위 즉시 DB 저장
# ============================================================
# 셀 5와 동일 구조 — 변수만 rail_* 로 분리 (셀 5의 failed/processed_total 보존)
# 1노선당 평균 ~18K건 → 페이지 18개씩 → 노선당 ~30~60초 예상

rail_failed: list[tuple[str, str, str]] = []
rail_started_at = datetime.now()
rail_processed_total = 0
rail_null_dropped = 0
rail_empty_lines = 0   # NO_DATA로 빈 응답 받은 노선 수

for i, row in enumerate(rail_remaining.itertuples(index=False), start=1):
    rte_id = row.rte_id
    rte_nm = row.rte_nm or "(이름없음)"

    try:
        items = fetch_all_pages(
            CARD_TRIPS_URL,
            CardTripItem,
            extra_params={
                "opr_ymd":       OPR_YMD,
                "rte_id":        rte_id,
                "users_type_cd": USERS_TYPE,
            },
        )

        if items:
            df_rows = pd.DataFrame([it.model_dump() for it in items])
            before_n = len(df_rows)
            df_rows = df_rows.dropna(subset=["vr_card_no", "ride_dt", "rte_id"])
            rail_null_dropped += before_n - len(df_rows)
            df_rows = df_rows.drop_duplicates(
                subset=["vr_card_no", "ride_dt", "rte_id"]
            )
        else:
            df_rows = pd.DataFrame()
            rail_empty_lines += 1

        if len(df_rows) > 0:
            with db_open() as con:
                con.register("df_view", df_rows)
                con.execute("""
                    INSERT INTO elderly_card_trips
                    SELECT vr_card_no, ride_dt, rte_id, opr_ymd, users_type_cd,
                           ride_ctpv_cd, goff_ctpv_cd, ride_sttn_id, goff_sttn_id, goff_dt,
                           clcln_bzmn_id, card_se_cd, clcln_bzmn_trfc_mns_cd,
                           trnf_cnt, utztn_nope, utztn_dstnc, brdg_hr, msv_intrpl_yn
                    FROM df_view
                    ON CONFLICT (vr_card_no, ride_dt, rte_id) DO NOTHING
                """)
                con.unregister("df_view")

        rail_processed_total += len(df_rows)
        status = f"+{len(df_rows):>6,}건" if len(df_rows) > 0 else "  (NO_DATA)"
    except Exception as e:
        rail_failed.append((rte_id, rte_nm, str(e)[:100]))
        status = f"FAILED ({type(e).__name__})"

    # 지하철은 노선 적으니 매 노선마다 출력
    elapsed = datetime.now() - rail_started_at
    pct = i / len(rail_remaining) * 100
    eta_sec = elapsed.total_seconds() / i * (len(rail_remaining) - i)
    print(f"[{i:2d}/{len(rail_remaining)}] {pct:5.1f}% | {rte_id:>4s} {rte_nm[:14]:14s} {status:14s} "
          f"| 누적 {rail_processed_total:>7,} | 경과 {str(elapsed).split('.')[0]} | 잔여 ~{int(eta_sec//60)}분")

    time.sleep(0.05)

print(f"\n✅ 지하철 수집 완료")
print(f"   이번 실행 적재: {rail_processed_total:,}건")
print(f"   NO_DATA 노선: {rail_empty_lines}개")
print(f"   실패 노선: {len(rail_failed)}개")
if rail_null_dropped > 0:
    print(f"   PK null 제외: {rail_null_dropped:,}건")
print(f"   총 소요시간: {datetime.now() - rail_started_at}")

with db_open(read_only=True) as con:
    grand_total = con.execute("SELECT COUNT(*) FROM elderly_card_trips").fetchone()[0]
print(f"\n📊 elderly_card_trips 총 누적 (버스+지하철): {grand_total:,}건")

[ 1/21]   4.8% |  004 4호선            +46,158건       | 누적  46,158 | 경과 0:02:59 | 잔여 ~59분
[ 2/21]   9.5% |  104 4호선              (NO_DATA)    | 누적  46,158 | 경과 0:03:00 | 잔여 ~28분
[ 3/21]  14.3% |  105 4호선              (NO_DATA)    | 누적  46,158 | 경과 0:03:00 | 잔여 ~18분
[ 4/21]  19.0% |  107 3호선              (NO_DATA)    | 누적  46,158 | 경과 0:03:01 | 잔여 ~12분
[ 5/21]  23.8% |  109 1호선              (NO_DATA)    | 누적  46,158 | 경과 0:03:01 | 잔여 ~9분
[ 6/21]  28.6% |  112 수인분당선            (NO_DATA)    | 누적  46,158 | 경과 0:03:01 | 잔여 ~7분
[ 7/21]  33.3% |  113 경강선              (NO_DATA)    | 누적  46,158 | 경과 0:03:02 | 잔여 ~6분
[ 8/21]  38.1% |  205 5호선            +56,334건       | 누적 102,492 | 경과 0:07:00 | 잔여 ~11분
[ 9/21]  42.9% |  206 6호선            +28,528건       | 누적 131,020 | 경과 0:09:02 | 잔여 ~12분
[10/21]  47.6% |  207 7호선            +46,502건       | 누적 177,522 | 경과 0:12:26 | 잔여 ~13분
[11/21]  52.4% |  301 인천1호선            (NO_DATA)    | 누적 177,522 | 경과 0:12:28 | 잔여 ~11분
[12/21]  57.1% |  302 인천2호선        

In [10]:
# ============================================================
# 셀 12 - 지하철 실패 노선 재시도
# ============================================================
if not rail_failed:
    print("실패 노선 없음")
else:
    print(f"재시도할 실패 노선: {len(rail_failed)}개")
    rail_retry_failed = []
    rail_retry_null_dropped = 0

    for rte_id, rte_nm, prev_err in rail_failed:
        try:
            items = fetch_all_pages(
                CARD_TRIPS_URL, CardTripItem,
                extra_params={"opr_ymd": OPR_YMD, "rte_id": rte_id, "users_type_cd": USERS_TYPE},
            )
            if items:
                df_rows = pd.DataFrame([it.model_dump() for it in items])
                before_n = len(df_rows)
                df_rows = df_rows.dropna(subset=["vr_card_no", "ride_dt", "rte_id"])
                rail_retry_null_dropped += before_n - len(df_rows)
                df_rows = df_rows.drop_duplicates(
                    subset=["vr_card_no", "ride_dt", "rte_id"]
                )

                if len(df_rows) > 0:
                    with db_open() as con:
                        con.register("df_view", df_rows)
                        con.execute("""
                            INSERT INTO elderly_card_trips
                            SELECT vr_card_no, ride_dt, rte_id, opr_ymd, users_type_cd,
                                   ride_ctpv_cd, goff_ctpv_cd, ride_sttn_id, goff_sttn_id, goff_dt,
                                   clcln_bzmn_id, card_se_cd, clcln_bzmn_trfc_mns_cd,
                                   trnf_cnt, utztn_nope, utztn_dstnc, brdg_hr, msv_intrpl_yn
                            FROM df_view
                            ON CONFLICT (vr_card_no, ride_dt, rte_id) DO NOTHING
                        """)
                        con.unregister("df_view")
                print(f"  ✅ {rte_id} {rte_nm[:15]}: +{len(df_rows)}건")
        except Exception as e:
            rail_retry_failed.append((rte_id, rte_nm, str(e)[:100]))
            print(f"  ❌ {rte_id}: {e}")
        time.sleep(0.2)

    rail_failed = rail_retry_failed
    print(f"\n재시도 후 여전히 실패: {len(rail_failed)}개")
    if rail_retry_null_dropped > 0:
        print(f"재시도 중 PK null 제외: {rail_retry_null_dropped:,}건")


실패 노선 없음
